In [6]:
import pandas as pd
import unidecode
from db_utils import crear_engine_mssql

Funciones Auxiliares

In [7]:
def limpiar_nombre_columna(nombre: str) -> str:
    """Normaliza nombres de columnas (sin tildes/espacios/puntos)."""
    nombre = unidecode.unidecode(str(nombre))
    return nombre.replace(" ", "_").replace(".", "_")

Mapeamos

In [8]:
engine = crear_engine_mssql()

Creacion de tabla bronce

In [9]:
# Rutas a datasets (desde raíz del proyecto)
arch1 = "Data/Bronce/2023.1_Sysarmy_Encuesta de remuneracin salarial Argentina.csv"
arch2 = "Data/Bronce/2023.2_Sysarmy_Encuesta de remuneracin salarial Argentina.csv"
arch3 = "Data/Bronce/Datos-Limpios-Sueldos2021.xlsx"
arch4 = "Data/Bronce/indice-salarios-base-2012-valores-mensuales.csv"

Carga de datasets

In [ ]:
df1 = pd.read_csv(arch1, encoding="utf-8")
df1.columns = [limpiar_nombre_columna(c) for c in df1.columns]
df1.to_sql("Sysarmy_2023_1", engine, schema="Bronce",
           if_exists="replace", index=False, chunksize=10000)

In [ ]:
df2 = pd.read_csv(arch2, encoding="utf-8")
df2.columns = [limpiar_nombre_columna(c) for c in df2.columns]
df2.to_sql("Sysarmy_2023_2", engine, schema="Bronce",
           if_exists="replace", index=False, chunksize=10000)

In [ ]:
df3 = pd.read_excel(arch3)
df3.columns = [limpiar_nombre_columna(c) for c in df3.columns]
df3.to_sql("Sueldos_2021", engine, schema="Bronce",
           if_exists="replace", index=False, chunksize=10000)

In [ ]:
df4 = pd.read_csv(arch4, encoding="latin-1")
df4.columns = [limpiar_nombre_columna(c) for c in df4.columns]
df4.to_sql("Indice_Salarios_Base2012", engine, schema="Bronce",
           if_exists="replace", index=False, chunksize=10000)

Revision de tablas

In [ ]:
q = """
SELECT s.name, t.name, SUM(p.rows) AS filas
FROM sys.tables t
JOIN sys.schemas s ON s.schema_id = t.schema_id
JOIN sys.partitions p ON t.object_id = p.object_id AND p.index_id IN (0,1)
WHERE s.name = 'Bronce'
GROUP BY s.name, t.name
ORDER BY t.name;
"""

df_check = pd.read_sql(q, engine)
df_check

,name,name,filas
0,Bronce,Indice_Salarios_Base2012,183
1,Bronce,Sueldos_2021,5876
2,Bronce,Sysarmy_2023_1,5767
3,Bronce,Sysarmy_2023_2,5422
